<a href="https://colab.research.google.com/github/bittu-quant-ai/bittu-quant-ai/blob/main/Anomalous_Intelligent_Expense_%26_Behavioral_Anomaly_Tracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# PHASE 1: ENVIRONMENT SETUP & IMPORTS
# ==========================================

# 1. Mount Google Drive for database persistence
from google.colab import drive

drive.mount("/content/drive")


print("Environment setup complete!")

Mounted at /content/drive
Environment setup complete!


In [2]:
# ==========================================
# STEP 1: INITIALIZE IN-MEMORY STORAGE
# ==========================================

# 1. Initialize an empty list to store our expenses in memory
expenses = []
print("Step 1 Complete: Storage initialized.")

Step 1 Complete: Storage initialized.


In [3]:

# ==========================================
# STEP 2: ADDING EXPENSES & CALCULATING MEAN
# ==========================================

# 2. Function to add a new expense to our list
def add_expense(amount, category,note=""):
    expense_record = {
         "amount": float(amount),
         "category": category.strip().capitalize(),
         "note": note,
    }
    expenses.append(expense_record)
    print(f"Added:${amount} under '{category}'")

In [4]:
# 2.1. Function to calculate the Average (Mean) for a specific category.
def calculate_category_mean(category_name):
    category_amounts = [
       exp["amount"]
      for exp in expenses
      if exp["category"] == category_name.capitalize()
   ]
# Edge case: If no expenses exist yet for this category, return 0
    if len(category_amounts) == 0:
       return 0.0
# Formula: Mean = Sum of amounts / Total count
    total_sum = sum(category_amounts)
    count = len(category_amounts)

    return total_sum / count
print("Step 2 Complete: Mean calculation logic ready.")

Step 2 Complete: Mean calculation logic ready.


In [5]:
# ==========================================
# STEP 3: SMART ANOMALY DETECTION
# ==========================================
def add_expense_with_check(amount, category, note=""):
   cat_cleaned = category.strip().capitalize()
   amount_float = float(amount)
#3.1. calculate the current average for this category BEFORE adding the new expense.
   current_avg = calculate_category_mean(cat_cleaned)
#3.2. Add the expense to our main list
   expense_record = {
      "amount": amount_float,
      "category": cat_cleaned,
      "note": note,
   }
   expenses.append(expense_record)
   print(f"Successfully added: ${amount_float} under '{cat_cleaned}'")

# 3.3. Anomaly Detection Logic
  # If we have history (average > 0) and the new amount is more than 2x the average:
   if current_avg > 0 and amount_float > (current_avg * 2):
     print(
        f"🚨 [ANOMALY DETECTED]: Your expense of ${amount_float} is significantly"
        f" higher than your usual average (${current_avg:.2f}) for"
        f" '{cat_cleaned}'!"
     )
   else:
     print(f"✓ Spending for '{cat_cleaned}' looks normal.\n")

In [6]:
# --- TEST THE FULL PIPELINE SO FAR ---
print("--- RUNNING TESTS ---")
add_expense_with_check(10.00, "Food", "Morning snack")
add_expense_with_check(12.00, "Food", "Lunch")
add_expense_with_check(
    45.00, "Food", "Expensive restaurant dinner (Should trigger anomaly)"
)

--- RUNNING TESTS ---
Successfully added: $10.0 under 'Food'
✓ Spending for 'Food' looks normal.

Successfully added: $12.0 under 'Food'
✓ Spending for 'Food' looks normal.

Successfully added: $45.0 under 'Food'
🚨 [ANOMALY DETECTED]: Your expense of $45.0 is significantly higher than your usual average ($11.00) for 'Food'!


In [7]:
print(expenses)

[{'amount': 10.0, 'category': 'Food', 'note': 'Morning snack'}, {'amount': 12.0, 'category': 'Food', 'note': 'Lunch'}, {'amount': 45.0, 'category': 'Food', 'note': 'Expensive restaurant dinner (Should trigger anomaly)'}]


In [8]:
# ==========================================
# STEP 4: INTERACTIVE USER INTERFACE LOOP
# ==========================================
print("--- STARTING EXPENSE TRACKER INTERACTIVE MENU ---")
print("Type your commands below:")
while True:
  action = (
      input("Type 'add' to add expense, 'view' to see all, or 'exit': ")
      .strip()
      .lower()
  )
  if action == "exit":
    print("Exiting Expense Tracker. Have a great day!")
    break
  elif action == "add":
    try:
      amount = float(input("Enter amount ($): "))
      category = input("Enter category (e.g., Food, Travel): ")
      note = input("Enter a short note: ")
# Call our smart function from Step 3 that checks for anomalies
      add_expense_with_check(amount, category, note)
    except ValueError:
      print(
          "⚠️ Invalid input! Please enter a valid number for the amount.\n"
      )
  elif action == "view":
    if len(expenses) == 0:
      print("No expenses recorded yet.\n")
    else:
      print("\n--- ALL RECORDED EXPENSES ---")
      for index, exp in enumerate(expenses, start=1):
        print(
            f"{index}. Amount: ${exp['amount']:.2f} | Category:"
            f" {exp['category']} | Note: {exp['note']}"
        )
      print("-" * 30 + "\n")
  else:
    print("⚠️ Unknown command. Please type 'add', 'view', or 'exit'.\n")

--- STARTING EXPENSE TRACKER INTERACTIVE MENU ---
Type your commands below:
Type 'add' to add expense, 'view' to see all, or 'exit': exit
Exiting Expense Tracker. Have a great day!


In [9]:
# ==========================================
# STEP 5: CATEGORY BUDGET CAPS & LIMIT ENFORCEMENT
# ==========================================
# 5.1. Define fixed monthly budget caps for 5-6 core categories
category_budgets = {
    "Food": 150.00,
    "Garments": 100.00,
    "Emi": 300.00,
    "Travel": 80.00,
    "Entertainment": 50.00,
}
# 5.2. Function to calculate total spent so far in a specific category
def calculate_category_total(category_name):
  cat_cleaned = category_name.strip().capitalize()
  category_amounts = [
      exp["amount"] for exp in expenses if exp["category"] == cat_cleaned
  ]
  return sum(category_amounts)
# 5.3. Enhanced add function incorporating Budget Limits + Anomaly Check
def add_expense_with_budget_check(amount, category, note=""):
  cat_cleaned = category.strip().capitalize()
  amount_float = float(amount)
# Check if the category has a defined budget limit
  if cat_cleaned not in category_budgets:
    print(
        f"⚠️ Warning: '{cat_cleaned}' has no budget limit set! Defaulting limit"
        " to $100.00"
    )
    budget_limit = 100.00
  else:
    budget_limit = category_budgets[cat_cleaned]
# Calculate current total spent in this category *before* adding this new one
  current_total = calculate_category_total(cat_cleaned)
  projected_total = current_total + amount_float

  # 5.4. Run the standard anomaly check (against historical average)
  current_avg = calculate_category_mean(cat_cleaned)
 # 5.5. Append expense to list
  expense_record = {
      "amount": amount_float,
      "category": cat_cleaned,
      "note": note,
  }
  expenses.append(expense_record)
  print(f"-> Added: ${amount_float:.2f} under '{cat_cleaned}'")
# 5.6. Budget Limit Enforcement Check
  if projected_total > budget_limit:
    print(
        f"🚨 [BUDGET EXCEEDED WARNING]: Adding this expense brings your total"
        f" for '{cat_cleaned}' to ${projected_total:.2f}, which crosses your"
        f" monthly limit of ${budget_limit:.2f}!"
    )
  elif projected_total >= (budget_limit * 0.8):
    print(
        f"⚠️ [CAUTION]: You have reached over 80% of your budget limit for"
        f" '{cat_cleaned}' (${projected_total:.2f} / ${budget_limit:.2f})."
    )
  else:
    print(f"✓ Budget status for '{cat_cleaned}': Safe.")
# 5.9. Anomaly Alert
  if current_avg > 0 and amount_float > (current_avg * 2):
    print(
        f"💡 [SPIKE NOTICE]: This single transaction (${amount_float:.2f}) is"
        f" double your historical average (${current_avg:.2f})."
    )

  print("-" * 40 + "\n")

In [10]:
# --- TEST THE NEW BUDGET ENFORCEMENT ---
print("--- TESTING BUDGET LIMITS ---")
add_expense_with_budget_check(100.00, "Food", "Groceries")  # Under $150 limit
add_expense_with_check(
    60.00, "Food", "More food"
)  # Will cross the $150 Food budget limit!

--- TESTING BUDGET LIMITS ---
-> Added: $100.00 under 'Food'
🚨 [BUDGET EXCEEDED WARNING]: Adding this expense brings your total for 'Food' to $167.00, which crosses your monthly limit of $150.00!
💡 [SPIKE NOTICE]: This single transaction ($100.00) is double your historical average ($22.33).
----------------------------------------

Successfully added: $60.0 under 'Food'
✓ Spending for 'Food' looks normal.

